# Data Analysis for SD-GPGPU2026 Submission, IV - Hierarchy Numbers 



### Data Preparation

The data is taken from the folder in Slack channel [silicon_mark](https://silicondata.slack.com/files/U09NQ432KSP/F0A4EBLQ7C4/siliconmark-provider-charts.zip).

Delete entries:
- GPU-0b8eb49d-d883-9cc1-aa43-749ef1afb730 is removed because it's way lower than the standard H100 PCIe.
- GPU-3369f751-3009-d274-8730-6484819eabb1 is removed because it's way lower than the standard A100 SXM4-80GB.



### Statistics by gpu_id

**Findings**
- Number of GPU ID: 3513
- Number of entries: 6809
- Number of GPU ID occurring more than 3 times: 745
- The maximum GPU fp16 spread: 8.37%
- The maximum GPU d2d_bw spread: 9.72%
- Percentage of GPUs with d2d_bw spread < 2%: 92.21%
- Percentage of GPUs with fp16 spread < 2%: 82.95%

**Conclusion**: The benchmark result is consistent.

In [1]:
%pip install pandas matplotlib seaborn scipy --quiet


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

df_raw = pd.read_csv('all_providers_data_2025-12-18.csv', sep=',')

Check if the gpu_id is same for different GPUs(different providers)

In [3]:
# Group by gpu_id and check consistency of provider and gpu_model
gpu_id_groups = df_raw.groupby('gpu_id').agg({
    'provider': lambda x: x.nunique(),
    'gpu_model': lambda x: x.nunique()
}).rename(columns={'provider': 'provider_count', 'gpu_model': 'gpu_model_count'})

# Find gpu_ids with multiple different providers or gpu_models
inconsistent = gpu_id_groups[(gpu_id_groups['provider_count'] > 1) | (gpu_id_groups['gpu_model_count'] > 1)]

print(f"Total number of gpu_ids: {len(gpu_id_groups)}")
print(f"Number of gpu_ids with inconsistencies: {len(inconsistent)}")
print("\nDetailed inconsistencies:")

# if len(inconsistent) > 0:
#     for idx, (gpu_id, row) in enumerate(inconsistent.iterrows()):
#         # print(f"\n--- gpu_id #{idx+1}: {gpu_id} ---")
#         # gpu_data = df_raw[df_raw['gpu_id'] == gpu_id][['provider', 'gpu_model']].drop_duplicates()
#         # print(f"Different provider/gpu_model combinations for this gpu_id:")
#         for i, data_row in gpu_data.iterrows():
#             # print(f"  provider: {data_row['provider']}, gpu_model: {data_row['gpu_model']}")
# else:
#     # print("No inconsistencies found. Each gpu_id corresponds to a unique provider and gpu_model.")

Total number of gpu_ids: 3513
Number of gpu_ids with inconsistencies: 21

Detailed inconsistencies:


In [4]:
# Check for gpu_model inconsistencies only
print("=" * 80)
print("Checking for GPU Model inconsistencies")
print("=" * 80)

# Find gpu_ids with gpu_model inconsistencies (gpu_model_count > 1)
only_gpu_model_inconsistent = gpu_id_groups[(gpu_id_groups['gpu_model_count'] > 1)]

print(f"\nNumber of gpu_ids with gpu_model inconsistencies: {len(only_gpu_model_inconsistent)}")

if len(only_gpu_model_inconsistent) > 0:
    print("\nDetailed inconsistencies:")
    for idx, (gpu_id, row) in enumerate(only_gpu_model_inconsistent.iterrows()):
        print(f"\n--- gpu_id #{idx+1}: {gpu_id} ---")
        gpu_data = df_raw[df_raw['gpu_id'] == gpu_id][['provider', 'gpu_model']].drop_duplicates()
        print(f"Different provider/gpu_model combinations for this gpu_id:")
        for i, data_row in gpu_data.iterrows():
            print(f"  provider: {data_row['provider']}, gpu_model: {data_row['gpu_model']}")
else:
    print("\nNo gpu_model inconsistencies found.")
    print("That is: No gpu_id corresponds to multiple different gpu_models.")

Checking for GPU Model inconsistencies

Number of gpu_ids with gpu_model inconsistencies: 0

No gpu_model inconsistencies found.
That is: No gpu_id corresponds to multiple different gpu_models.


In [5]:
# Add suffix "_1" to duplicate gpu_ids from different providers
# For gpu_ids that appear in multiple providers, add "_1" to non-first providers

def add_suffix_to_duplicate_gpu_ids(df):
    """
    For each gpu_id that appears in multiple providers,
    add "_1" suffix to gpu_ids from the second and subsequent providers.
    
    The first occurrence of each gpu_id keeps the original name.
    """
    df_modified = df.copy()
    
    # Find gpu_ids that appear with multiple providers
    gpu_id_providers = df_modified.groupby('gpu_id')['provider'].nunique()
    duplicate_gpu_ids = gpu_id_providers[gpu_id_providers > 1].index
    
    # print(f"Total gpu_ids with multiple providers: {len(duplicate_gpu_ids)}")
    
    # For each duplicate gpu_id, mark which provider gets the suffix
    for gpu_id in duplicate_gpu_ids:
        mask = df_modified['gpu_id'] == gpu_id
        rows_with_gpu_id = df_modified[mask].copy()
        
        # Sort by provider to ensure consistent ordering
        rows_with_gpu_id = rows_with_gpu_id.sort_values('provider')
        providers_list = rows_with_gpu_id['provider'].unique()
        
        # print(f"\ngpu_id: {gpu_id}")
        # print(f"  Providers: {list(providers_list)}")
        
        # For each provider except the first, add "_1" suffix
        for idx, provider in enumerate(providers_list):
            if idx > 0:  # Skip the first provider
                # Find rows with this gpu_id and provider, and add suffix
                mask_for_suffix = (df_modified['gpu_id'] == gpu_id) & (df_modified['provider'] == provider)
                df_modified.loc[mask_for_suffix, 'gpu_id'] = gpu_id + '_1'
                # print(f"    {provider}: gpu_id changed to {gpu_id}_1")
    
    return df_modified

# Apply the function
df_raw = add_suffix_to_duplicate_gpu_ids(df_raw)

print("\n" + "="*80)
print("Data processing completed!")
print(f"Original shape: {df_raw.shape}")
print(f"Processed shape: {df_raw.shape}")
print("\nSample of modified gpu_ids:")
print(df_raw[['provider', 'gpu_id', 'gpu_model']].drop_duplicates('gpu_id').head(25))


Data processing completed!
Original shape: (6808, 12)
Processed shape: (6808, 12)

Sample of modified gpu_ids:
   provider                                    gpu_id              gpu_model
0    lambda  GPU-9913471a-ff1c-457a-ef1e-b5ed15d5f390  NVIDIA A100-SXM4-80GB
1    lambda  GPU-3b970437-7309-e356-d213-2b9ed61ffa9b  NVIDIA A100-SXM4-80GB
2    lambda  GPU-b820db02-ea9f-32d3-be3c-8b294cb5d2f0  NVIDIA A100-SXM4-80GB
3    lambda  GPU-f32a0860-e1f8-8a74-5cb5-376af6b0834c  NVIDIA A100-SXM4-80GB
4    lambda  GPU-7ca64762-23f1-2aa5-42cb-26c3a0c18ef5  NVIDIA A100-SXM4-80GB
5    lambda  GPU-d9f291b1-0f87-2227-b427-14e0b3aee52d  NVIDIA A100-SXM4-80GB
6    lambda  GPU-0d4869b4-975c-304c-4978-1c15c6072b84  NVIDIA A100-SXM4-80GB
7    lambda  GPU-328ecf89-8788-d274-e04b-b61d2b639099  NVIDIA A100-SXM4-80GB
8    lambda  GPU-e926bdaf-6197-1b34-9b55-879228cf187f  NVIDIA A100-SXM4-80GB
9    lambda  GPU-f6c128ea-0432-4ee6-de33-a322d945191c  NVIDIA A100-SXM4-80GB
10   lambda  GPU-91ae3077-5ac4-d048-011e-

In [6]:
# Group by gpu_id and check consistency of provider and gpu_model
gpu_id_groups = df_raw.groupby('gpu_id').agg({
    'provider': lambda x: x.nunique(),
    'gpu_model': lambda x: x.nunique()
}).rename(columns={'provider': 'provider_count', 'gpu_model': 'gpu_model_count'})

# Find gpu_ids with multiple different providers or gpu_models
inconsistent = gpu_id_groups[(gpu_id_groups['provider_count'] > 1) | (gpu_id_groups['gpu_model_count'] > 1)]

print(f"Total number of gpu_ids: {len(gpu_id_groups)}")
print(f"Number of gpu_ids with inconsistencies: {len(inconsistent)}")
print("\nDetailed inconsistencies:")

if len(inconsistent) > 0:
    for idx, (gpu_id, row) in enumerate(inconsistent.iterrows()):
        print(f"\n--- gpu_id #{idx+1}: {gpu_id} ---")
        gpu_data = df_raw[df_raw['gpu_id'] == gpu_id][['provider', 'gpu_model']].drop_duplicates()
        print(f"Different provider/gpu_model combinations for this gpu_id:")
        for i, data_row in gpu_data.iterrows():
            print(f"  provider: {data_row['provider']}, gpu_model: {data_row['gpu_model']}")
else:
    print("No inconsistencies found. Each gpu_id corresponds to a unique provider and gpu_model.")

Total number of gpu_ids: 3534
Number of gpu_ids with inconsistencies: 0

Detailed inconsistencies:
No inconsistencies found. Each gpu_id corresponds to a unique provider and gpu_model.


In [7]:
gpu_num = df_raw['gpu_id'].nunique()
print(f'The number of unique GPU id is: {gpu_num}')

gpu_id_counts = df_raw["gpu_id"].value_counts()

# gpu_id with the maximum occurrences
top_gpu_id = gpu_id_counts.idxmax()
top_gpu_count = gpu_id_counts.max()

print(f"GPU ID with the most occurrences: {top_gpu_id}")
print(f"Number of occurrences: {top_gpu_count}")

The number of unique GPU id is: 3534
GPU ID with the most occurrences: GPU-cd775363-fab7-e3d8-4d61-610a609c22f5
Number of occurrences: 12


- The number of unique GPU id is: 3534.
- GPU ID with the most occurrences: GPU-713b4618-fb55-60a6-c431-6c9fb26e6359; Max number of occurrences: 12.

Find the GPUs occur more than 3 times. Calculate its fp16 max/avg, min/avg ratio spread and plot.

In [8]:
# --- Ensure numeric columns are numeric (coerce bad values to NaN) ---
for col in ["fp16", "memory_bandwidth"]:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")


In [9]:
provider_anon_dict = {
    'aws': 'Cld-1',
    'gcp': 'Cld-5',
    'boostrun': 'Cld-2',
    'crusoe': 'Cld-3',
    'datacrunch': 'Cld-4',
    'lambda': 'Cld-6',
    'massedcompute': 'Cld-9',
    'nebius': 'Cld-7',
    'paperspace': 'Cld-8',
    'runpod': 'Cld-9'
}
df_raw["provider"] = df_raw["provider"].map(provider_anon_dict)

## Device level

In [10]:
gpu_keep_map = {
    'NVIDIA A100 80GB PCIe': 'A100 PCIe',
    'NVIDIA A100-SXM4-80GB': 'A100 SXM4',
    'NVIDIA H100 PCIe': 'H100 PCIe',
    'NVIDIA H100 80GB HBM3': 'H100 HBM3',
    'NVIDIA H200': 'H200 SXM',
}

FP16_NORM_COL = "fp16"   # <-- change if needed
BW_NORM_COL   = "memory_bandwidth"     # <-- change if needed

# --- 1) Keep only target GPU models ---
df_tgt = df_raw[df_raw["gpu_model"].isin(gpu_keep_map.keys())].copy()
df_tgt["gpu_model_short"] = df_tgt["gpu_model"].map(gpu_keep_map)

# --- 2) Only keep gpu_id with occurrences >= 3 (within the filtered target set) ---
counts = df_tgt["gpu_id"].value_counts()
valid_gpu_ids = counts[counts >= 3].index
df_filt = df_tgt[df_tgt["gpu_id"].isin(valid_gpu_ids)].copy()

# --- 3) Aggregate per-device stats and compute (max-min)/mean for FP16_norm and BW_norm ---
stats = (
    df_filt.groupby("gpu_id")
    .agg(
        occurrences=("gpu_id", "count"),
        provider=("provider", "first"),
        gpu_model=("gpu_model", "first"),
        gpu_model_short=("gpu_model_short", "first"),

        fp16n_min=(FP16_NORM_COL, "min"),
        fp16n_max=(FP16_NORM_COL, "max"),
        fp16n_mean=(FP16_NORM_COL, "mean"),

        bwn_min=(BW_NORM_COL, "min"),
        bwn_max=(BW_NORM_COL, "max"),
        bwn_mean=(BW_NORM_COL, "mean"),
    )
    .reset_index()
)

# Guard against mean==0 (shouldn't happen for normalized metrics, but safe)
stats["fp16_norm_range"] = (stats["fp16n_max"] - stats["fp16n_min"]) / stats["fp16n_mean"].replace(0, pd.NA)
stats["bw_norm_range"]   = (stats["bwn_max"]   - stats["bwn_min"])   / stats["bwn_mean"].replace(0, pd.NA)


model_summary = (
    stats.groupby("gpu_model_short")
    .agg(
        n_devices=("gpu_id", "nunique"),
        fp16_range_median=("fp16_norm_range", "median"),
        fp16_range_p90=("fp16_norm_range", lambda s: s.quantile(0.9)),
        fp16_range_max=("fp16_norm_range", "max"),
        bw_range_median=("bw_norm_range", "median"),
        bw_range_p90=("bw_norm_range", lambda s: s.quantile(0.9)),
        bw_range_max=("bw_norm_range", "max"),
    )
    .reset_index()
)

model_summary.round(4)



,gpu_model_short,n_devices,fp16_range_median,fp16_range_p90,fp16_range_max,bw_range_median,bw_range_p90,bw_range_max
0,A100 PCIe,32,0.0147,0.0293,0.0386,0.0020,0.0038,0.0185
1,A100 SXM4,321,0.0060,0.0120,0.0256,0.0007,0.0012,0.0865
2,H100 HBM3,53,0.0170,0.0457,0.0573,0.0019,0.0112,0.0256
3,H100 PCIe,30,0.0140,0.0550,0.0837,0.0018,0.0043,0.0118
4,H200 SXM,25,0.0123,0.0206,0.0278,0.0208,0.0472,0.0534


## Intra-provider level

In [11]:
import pandas as pd

df_raw = pd.read_csv('all_providers_data_2025-12-18.csv', sep=',')

In [12]:
# Check for gpu_model inconsistencies only
print("=" * 80)
print("Checking for GPU Model inconsistencies")
print("=" * 80)

# Find gpu_ids with gpu_model inconsistencies (gpu_model_count > 1)
only_gpu_model_inconsistent = gpu_id_groups[(gpu_id_groups['gpu_model_count'] > 1)]

print(f"\nNumber of gpu_ids with gpu_model inconsistencies: {len(only_gpu_model_inconsistent)}")

if len(only_gpu_model_inconsistent) > 0:
    print("\nDetailed inconsistencies:")
    for idx, (gpu_id, row) in enumerate(only_gpu_model_inconsistent.iterrows()):
        print(f"\n--- gpu_id #{idx+1}: {gpu_id} ---")
        gpu_data = df_raw[df_raw['gpu_id'] == gpu_id][['provider', 'gpu_model']].drop_duplicates()
        print(f"Different provider/gpu_model combinations for this gpu_id:")
        for i, data_row in gpu_data.iterrows():
            print(f"  provider: {data_row['provider']}, gpu_model: {data_row['gpu_model']}")
else:
    print("\nNo gpu_model inconsistencies found.")
    print("That is: No gpu_id corresponds to multiple different gpu_models.")

Checking for GPU Model inconsistencies

Number of gpu_ids with gpu_model inconsistencies: 0

No gpu_model inconsistencies found.
That is: No gpu_id corresponds to multiple different gpu_models.


In [13]:
# Add suffix "_1" to duplicate gpu_ids from different providers
# For gpu_ids that appear in multiple providers, add "_1" to non-first providers

def add_suffix_to_duplicate_gpu_ids(df):
    """
    For each gpu_id that appears in multiple providers,
    add "_1" suffix to gpu_ids from the second and subsequent providers.
    
    The first occurrence of each gpu_id keeps the original name.
    """
    df_modified = df.copy()
    
    # Find gpu_ids that appear with multiple providers
    gpu_id_providers = df_modified.groupby('gpu_id')['provider'].nunique()
    duplicate_gpu_ids = gpu_id_providers[gpu_id_providers > 1].index
    
    # print(f"Total gpu_ids with multiple providers: {len(duplicate_gpu_ids)}")
    
    # For each duplicate gpu_id, mark which provider gets the suffix
    for gpu_id in duplicate_gpu_ids:
        mask = df_modified['gpu_id'] == gpu_id
        rows_with_gpu_id = df_modified[mask].copy()
        
        # Sort by provider to ensure consistent ordering
        rows_with_gpu_id = rows_with_gpu_id.sort_values('provider')
        providers_list = rows_with_gpu_id['provider'].unique()
        
        # print(f"\ngpu_id: {gpu_id}")
        # print(f"  Providers: {list(providers_list)}")
        
        # For each provider except the first, add "_1" suffix
        for idx, provider in enumerate(providers_list):
            if idx > 0:  # Skip the first provider
                # Find rows with this gpu_id and provider, and add suffix
                mask_for_suffix = (df_modified['gpu_id'] == gpu_id) & (df_modified['provider'] == provider)
                df_modified.loc[mask_for_suffix, 'gpu_id'] = gpu_id + '_1'
                # print(f"    {provider}: gpu_id changed to {gpu_id}_1")
    
    return df_modified

# Apply the function
df_raw = add_suffix_to_duplicate_gpu_ids(df_raw)

print("\n" + "="*80)
print("Data processing completed!")
print(f"Original shape: {df_raw.shape}")
print(f"Processed shape: {df_raw.shape}")
print("\nSample of modified gpu_ids:")
print(df_raw[['provider', 'gpu_id', 'gpu_model']].drop_duplicates('gpu_id').head(25))


Data processing completed!
Original shape: (6808, 12)
Processed shape: (6808, 12)

Sample of modified gpu_ids:
   provider                                    gpu_id              gpu_model
0    lambda  GPU-9913471a-ff1c-457a-ef1e-b5ed15d5f390  NVIDIA A100-SXM4-80GB
1    lambda  GPU-3b970437-7309-e356-d213-2b9ed61ffa9b  NVIDIA A100-SXM4-80GB
2    lambda  GPU-b820db02-ea9f-32d3-be3c-8b294cb5d2f0  NVIDIA A100-SXM4-80GB
3    lambda  GPU-f32a0860-e1f8-8a74-5cb5-376af6b0834c  NVIDIA A100-SXM4-80GB
4    lambda  GPU-7ca64762-23f1-2aa5-42cb-26c3a0c18ef5  NVIDIA A100-SXM4-80GB
5    lambda  GPU-d9f291b1-0f87-2227-b427-14e0b3aee52d  NVIDIA A100-SXM4-80GB
6    lambda  GPU-0d4869b4-975c-304c-4978-1c15c6072b84  NVIDIA A100-SXM4-80GB
7    lambda  GPU-328ecf89-8788-d274-e04b-b61d2b639099  NVIDIA A100-SXM4-80GB
8    lambda  GPU-e926bdaf-6197-1b34-9b55-879228cf187f  NVIDIA A100-SXM4-80GB
9    lambda  GPU-f6c128ea-0432-4ee6-de33-a322d945191c  NVIDIA A100-SXM4-80GB
10   lambda  GPU-91ae3077-5ac4-d048-011e-

In [14]:
gpu_num = df_raw['gpu_id'].nunique()
print(f'The number of unique GPU id is: {gpu_num}')

gpu_id_counts = df_raw["gpu_id"].value_counts()

# gpu_id with the maximum occurrences
top_gpu_id = gpu_id_counts.idxmax()
top_gpu_count = gpu_id_counts.max()

print(f"GPU ID with the most occurrences: {top_gpu_id}")
print(f"Number of occurrences: {top_gpu_count}")

The number of unique GPU id is: 3534
GPU ID with the most occurrences: GPU-cd775363-fab7-e3d8-4d61-610a609c22f5
Number of occurrences: 12


In [15]:
# --- Ensure numeric columns are numeric (coerce bad values to NaN) ---
for col in ["fp16", "memory_bandwidth"]:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")


In [16]:
provider_anon_dict = {
    'aws': 'Cld-1',
    'gcp': 'Cld-5',
    'boostrun': 'Cld-2',
    'crusoe': 'Cld-3',
    'datacrunch': 'Cld-4',
    'lambda': 'Cld-6',
    'massedcompute': 'Cld-9',
    'nebius': 'Cld-7',
    'paperspace': 'Cld-8',
    'runpod': 'Cld-9'
}
df_raw["provider"] = df_raw["provider"].map(provider_anon_dict)

gpu_keep_map = {
    'NVIDIA A100 80GB PCIe': 'A100 PCIe',
    'NVIDIA A100-SXM4-80GB': 'A100 SXM4',
    'NVIDIA H100 PCIe': 'H100 PCIe',
    'NVIDIA H100 80GB HBM3': 'H100 HBM3',
    'NVIDIA H200': 'H200',
}

df_filt = df_raw[df_raw['gpu_model'].isin(gpu_keep_map)].copy()

In [17]:
# --- Group by GPU model and provider and provider to get performance metrics ---
provider_gpu_stats = (
    df_filt.groupby(['gpu_model', 'provider'])
    .agg(
        occurrences=('gpu_id', 'count'),
        fp16_min=('fp16', 'min'),
        fp16_max=('fp16', 'max'),
        fp16_avg=('fp16', 'mean'),
        bw_min=('memory_bandwidth', 'min'),
        bw_max=('memory_bandwidth', 'max'),
        bw_avg=('memory_bandwidth', 'mean'),
    )
    .reset_index()
)

# --- Calculate normalized percentage range ---
provider_gpu_stats['bw_norm_range'] = 100 * \
    (provider_gpu_stats['bw_max'] - provider_gpu_stats['bw_min']) / provider_gpu_stats['bw_avg']
provider_gpu_stats['fp16_norm_range'] = 100 * \
    (provider_gpu_stats['fp16_max'] - provider_gpu_stats['fp16_min']) / provider_gpu_stats['fp16_avg']

# --- Display the results ---
display_cols = ['gpu_model', 'provider', 'occurrences', 
                'fp16_norm_range', 'bw_norm_range']
display(provider_gpu_stats
        .loc[provider_gpu_stats["occurrences"] >= 3, display_cols]
        .round(2))

,gpu_model,provider,occurrences,fp16_norm_range,bw_norm_range
0,NVIDIA A100 80GB PCIe,Cld-3,39,6.88,0.39
1,NVIDIA A100 80GB PCIe,Cld-9,215,9.73,12.70
2,NVIDIA A100-SXM4-80GB,Cld-4,117,4.76,0.86
3,NVIDIA A100-SXM4-80GB,Cld-5,3,1.14,0.04
4,NVIDIA A100-SXM4-80GB,Cld-6,1616,8.71,0.51
5,NVIDIA A100-SXM4-80GB,Cld-8,24,6.18,0.31
6,NVIDIA A100-SXM4-80GB,Cld-9,336,18.70,17.51
7,NVIDIA H100 80GB HBM3,Cld-4,116,7.64,1.12
8,NVIDIA H100 80GB HBM3,Cld-5,124,5.67,16.40
9,NVIDIA H100 80GB HBM3,Cld-6,66,8.96,0.44


In [18]:
max_range_by_model = (
    provider_gpu_stats
    .groupby("gpu_model")
    .agg(
        max_fp16_norm_range=("fp16_norm_range", "max"),
        max_bw_norm_range=("bw_norm_range", "max"),
        n_providers=("provider", "nunique"),
    )
    .reset_index()
)

display(max_range_by_model.round(2))


,gpu_model,max_fp16_norm_range,max_bw_norm_range,n_providers
0,NVIDIA A100 80GB PCIe,9.73,12.70,2
1,NVIDIA A100-SXM4-80GB,18.70,17.51,5
2,NVIDIA H100 80GB HBM3,9.58,16.95,6
3,NVIDIA H100 PCIe,32.53,10.07,3
4,NVIDIA H200,6.97,38.14,4


## Inter-Provider level

In [19]:
# --- Group by GPU model and provider to get performance metrics ---
provider_gpu_stats = (
    df_filt.groupby(['gpu_model'])
    .agg(
        occurrences=('gpu_id', 'count'),
        fp16_min=('fp16', 'min'),
        fp16_max=('fp16', 'max'),
        fp16_avg=('fp16', 'mean'),
        bw_min=('memory_bandwidth', 'min'),
        bw_max=('memory_bandwidth', 'max'),
        bw_avg=('memory_bandwidth', 'mean'),
    )
    .reset_index()
)

# --- Calculate normalized percentage range ---
provider_gpu_stats['bw_norm_range'] = 100 * \
    (provider_gpu_stats['bw_max'] - provider_gpu_stats['bw_min']) / provider_gpu_stats['bw_avg']
provider_gpu_stats['fp16_norm_range'] = 100 * \
    (provider_gpu_stats['fp16_max'] - provider_gpu_stats['fp16_min']) / provider_gpu_stats['fp16_avg']

# --- Display the results ---
display_cols = ['gpu_model', 'occurrences', 
                'fp16_norm_range', 'bw_norm_range']
display(provider_gpu_stats[display_cols].round(2))

,gpu_model,occurrences,fp16_norm_range,bw_norm_range
0,NVIDIA A100 80GB PCIe,254,9.70,12.64
1,NVIDIA A100-SXM4-80GB,2096,19.12,17.19
2,NVIDIA H100 80GB HBM3,452,10.40,16.67
3,NVIDIA H100 PCIe,259,34.49,10.08
4,NVIDIA H200,253,7.12,37.82
